# Face and Eye Detection

## Aim
Develop a Haar-cascade face/eye detector and evaluate detection under different lighting conditions.

## Theory
Haar cascades evaluate rectangular intensity features through a boosted classifier. Histogram equalization reduces some illumination variation. The evaluation uses one known face per image, so face recall is 1 when at least one face is found.

In [ ]:
!pip -q install opencv-python-headless matplotlib pandas

In [ ]:
# Portable setup: local Jupyter, Google Drive, or Google Colab
from pathlib import Path
import os, zipfile

FOLDER_NAME = "05_PRACTICAL_5"
ZIP_NAME = "05_PRACTICAL_5.zip"

def locate_project():
    candidates = [Path.cwd(), Path.cwd()/FOLDER_NAME, Path('/content')/FOLDER_NAME]
    candidates += list(Path('/content').glob('**/' + FOLDER_NAME)) if Path('/content').exists() else []
    for p in candidates:
        if (p/'inputs').exists() and (p/'outputs').exists():
            return p.resolve()
    return None

BASE = locate_project()
if BASE is None:
    try:
        from google.colab import files
        print(f"Upload {ZIP_NAME} when the chooser opens.")
        uploaded = files.upload()
        chosen = next((n for n in uploaded if n.lower().endswith('.zip')), None)
        if chosen:
            with zipfile.ZipFile(chosen) as zf:
                zf.extractall('/content')
            BASE = locate_project()
    except ImportError:
        pass
if BASE is None:
    raise FileNotFoundError(
        f"Could not find {FOLDER_NAME}. Extract {ZIP_NAME}, or upload that ZIP in Colab."
    )
os.chdir(BASE)
INPUTS, OUTPUTS, MODELS = BASE/'inputs', BASE/'outputs', BASE/'models'
OUTPUTS.mkdir(exist_ok=True)
print('Project directory:', BASE)


In [ ]:
import cv2, pandas as pd, matplotlib.pyplot as plt
face=cv2.CascadeClassifier(str(MODELS/'haarcascade_frontalface_default.xml'));eye=cv2.CascadeClassifier(str(MODELS/'haarcascade_eye_tree_eyeglasses.xml'))
rows=[];views=[]
for lighting in ['normal','bright','dim']:
 img=cv2.imread(str(INPUTS/f'face_{lighting}.jpg'));gray=cv2.equalizeHist(cv2.cvtColor(img,cv2.COLOR_BGR2GRAY));faces=face.detectMultiScale(gray,1.1,5,minSize=(60,60));eye_count=0;out=img.copy()
 for x,y,w,h in faces:
  cv2.rectangle(out,(x,y),(x+w,y+h),(0,255,0),2);eyes=eye.detectMultiScale(gray[y:y+h,x:x+w],1.1,6,minSize=(18,18));eye_count+=len(eyes)
  for ex,ey,ew,eh in eyes[:2]:cv2.rectangle(out,(x+ex,y+ey),(x+ex+ew,y+ey+eh),(255,0,0),2)
 cv2.imwrite(str(OUTPUTS/f'detected_{lighting}_rerun.jpg'),out);rows.append([lighting,len(faces),eye_count,1 if faces is not None and len(faces)>=1 else 0]);views.append(out)
df=pd.DataFrame(rows,columns=['lighting','faces_detected','eyes_detected','face_recall']);display(df);df.to_csv(OUTPUTS/'lighting_accuracy_rerun.csv',index=False)
fig,ax=plt.subplots(1,3,figsize=(12,4))
for a,x,t in zip(ax,views,['Normal','Bright','Dim']):a.imshow(cv2.cvtColor(x,cv2.COLOR_BGR2RGB));a.set_title(t);a.axis('off')
plt.show()

## Result and conclusion
Run all cells and compare the generated files in `outputs/` with the supplied reference outputs. The observations and conclusion are also summarized in `OUTPUT_SUMMARY.md`.